In [3]:
"""
causal_shap_drbo.py
====================
Causal SHAP feature attribution using DrBO (DAG Recovery via Bayesian Optimization)
for causal graph discovery across 6 datasets.

Dataset summary (from actual files):
  yield_preprocessed.npz     : 1000 samples, 4 features,  3-class classification
  soil_spi_preprocessed.npz  : 5000 samples, 7 features,  3-class classification
  cv_risk.csv                 : 1000 samples, 4 features,  regression (continuous)
  lung_cancer_risk.csv        : 1000 samples, 3 features,  regression (continuous)
  wine_preprocessed.npz       :  177 samples, 13 features, 3-class classification
  compas_preprocessed.npz     : 5278 samples, 19 features, binary classification

Setup (run once on the server):
  git clone https://github.com/baosws/DrBO.git
  pip install -r DrBO/requirements.txt
  pip install shap scikit-learn pandas numpy scipy openpyxl networkx matplotlib

Usage:
  python causal_shap_drbo.py [--data_dir /path/to/datasets]

Outputs written to ./results_drbo/
"""

import os, sys, time, warnings, argparse
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.multiclass import OneVsRestClassifier
import networkx as nx

warnings.filterwarnings("ignore")

parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", default="../../../datasets",
                    help="Directory containing the dataset files")
parser.add_argument("--n_evals", type=int, default=20000,
                    help="Total DrBO DAG evaluations (lower = faster, less accurate)")
parser.add_argument("--n_shap_samples", type=int, default=200,
                    help="Number of samples to compute SHAP on")
parser.add_argument("--M", type=int, default=512,
                    help="Permutations per SHAP sample")
parser.add_argument("--seed", type=int, default=42)
args, _ = parser.parse_known_args()

DATA_DIR = args.data_dir
OUT_DIR      = "results_drbo"
BARCHART_DIR = os.path.join(OUT_DIR, "bar_charts")
GRAPH_DIR    = os.path.join(OUT_DIR, "graphs")
for d in [OUT_DIR, BARCHART_DIR, GRAPH_DIR]:
    os.makedirs(d, exist_ok=True)

# 1. Environment-agnostic path safety for Jupyter Notebooks vs .py scripts
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

DRBO_ROOT = os.path.join(SCRIPT_DIR, "DrBO")
if os.path.isdir(DRBO_ROOT):
    sys.path.insert(0, DRBO_ROOT)
else:
    LOCAL_DRBO = os.path.join(os.getcwd(), "DrBO")
    if os.path.isdir(LOCAL_DRBO):
        sys.path.insert(0, LOCAL_DRBO)
        DRBO_ROOT = LOCAL_DRBO

# 3. Attempt to fetch DrBO objects safely
try:
    from drbo import DrBO
    from drbo.pruners import prune_linear, prune_cit
    DRBO_AVAILABLE = True
    print("[OK] DrBO packages imported successfully in Jupyter environment.")
except ImportError:
    DRBO_AVAILABLE = False
    print(f"[WARN] Could not find DrBO code at: {DRBO_ROOT}\n"
          f"Please verify that the 'DrBO' folder is in your current working directory:\n"
          f" -> {os.getcwd()}")

if os.path.isdir(DRBO_ROOT):
    sys.path.insert(0, DRBO_ROOT)
else:
    LOCAL_DRBO = os.path.join(os.getcwd(), "DrBO")
    if os.path.isdir(LOCAL_DRBO):
        sys.path.insert(0, LOCAL_DRBO)
if os.path.isdir(DRBO_ROOT):
    sys.path.insert(0, DRBO_ROOT)

try:
    from drbo import DrBO
    from drbo.score import BIC_EV, BIC_NV
    DRBO_AVAILABLE = True
    print("[OK] DrBO found.")
except ImportError:
    DRBO_AVAILABLE = False
    print("[WARN] DrBO not found — will use DAGMA Linear fallback.\n"
          "       Install: git clone https://github.com/baosws/DrBO.git\n"
          "                pip install -r DrBO/requirements.txt")

try:
    from dagma.linear import DagmaLinear
    DAGMA_AVAILABLE = True
    print("[OK] DAGMA found (fallback ready).")
except ImportError:
    DAGMA_AVAILABLE = False


# Dataset loaders

def load_npz(filename):
    """Load a preprocessed .npz file, merging train/test splits."""
    path = os.path.join(DATA_DIR, filename)
    d = np.load(path, allow_pickle=True)
    X = np.concatenate([d["X_train"], d["X_test"]], axis=0)
    y = np.concatenate([d["y_train"], d["y_test"]], axis=0)
    feat_names = [str(f) for f in d["feature_names"]]
    return X, y, feat_names


def load_yield():
    X, y, feat_names = load_npz("yield_preprocessed.npz")
    # y ∈ {0,1,2} - multiclass classification
    return X, y, None, feat_names, "multiclass", "yield"


def load_soil():
    X, y, feat_names = load_npz("soil_spi_preprocessed.npz")
    # y ∈ {0,1,2} - multiclass classification
    return X, y, None, feat_names, "multiclass", "soil"


def load_cv_risk():
    df = pd.read_csv(os.path.join(DATA_DIR, "cv_risk.csv"))
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c])
    target_col = "cv_risk"
    feat_names = [c for c in df.columns if c != target_col]
    X = df[feat_names].values.astype(float)
    y = df[target_col].values.astype(float)
    # continuous target - regression
    return X, y, None, feat_names, "regression", "cv_risk"


def load_lung_cancer():
    df = pd.read_csv(os.path.join(DATA_DIR, "lung_cancer_risk.csv"))
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c])
    target_col = "lung_cancer_risk"
    feat_names = [c for c in df.columns if c != target_col]
    X = df[feat_names].values.astype(float)
    y = df[target_col].values.astype(float)
    return X, y, None, feat_names, "regression", "lung_cancer"


def load_wine():
    X, y, feat_names = load_npz("wine_preprocessed.npz")
    # y ∈ {1,2,3} - 3-class classification
    return X, y, None, feat_names, "multiclass", "wine"


def load_compas():
    X, y, feat_names = load_npz("compas_preprocessed.npz")
    # y ∈ {0,1} - binary classification
    return X, y, None, feat_names, "binary", "compas"


DATASET_LOADERS = [
    load_yield,
    load_soil,
    load_cv_risk,
    load_lung_cancer,
    load_wine,
    load_compas,
]


# Causal discovery

def run_drbo(X_scaled: np.ndarray, task_type: str, dataset_name: str) -> tuple:
    """
    Run DrBO on standardised feature matrix X_scaled.
    Returns:
        adj_weighted  (d×d) float — weighted adjacency, [i,j] = weight of edge i→j
        elapsed_sec   float
    """
    d = X_scaled.shape[1]
    # Use BIC-NV for all tasks (more general; handles unequal noise variances)
    score_fn = BIC_NV(X_scaled)

    t0 = time.time()
    model = DrBO(
        score_fn   = score_fn,
        dag_rank   = min(8, d - 1),   # k - low-rank DAG representation
        batch_size = 64,              # B
        n_prelim   = 100_000,         # C - preliminary candidates per step
        n_evals    = args.n_evals,    # T - total DAG evaluations
        n_grads    = 10,              # gradient steps per BO iteration
        n_replay   = 1024,
        dropout    = 0.1,
        n_hidden   = 64,
        lr         = 0.1,
        seed       = args.seed,
        verbose    = True,
    )
    model.fit(X_scaled)
    elapsed = time.time() - t0

    adj_binary = model.dag_   # (d×d) int, adj_binary[i,j]=1 -> edge i->j

    # Recover continuous edge weights via OLS on each node's parents
    adj_weighted = np.zeros((d, d), dtype=float)
    for j in range(d):
        parents = np.where(adj_binary[:, j] == 1)[0]
        if len(parents) == 0:
            continue
        reg = Ridge(alpha=1e-3, fit_intercept=True)
        reg.fit(X_scaled[:, parents], X_scaled[:, j])
        for k_idx, p in enumerate(parents):
            adj_weighted[p, j] = float(reg.coef_[k_idx])

    n_edges = int(adj_binary.sum())
    print(f"  [DrBO] {dataset_name}: {n_edges} edges in {elapsed:.1f}s")
    return adj_weighted, elapsed


def run_dagma_fallback(X_scaled: np.ndarray, dataset_name: str) -> tuple:
    """DAGMA Linear fallback."""
    print(f"  [DAGMA fallback] {dataset_name}")
    t0 = time.time()
    model = DagmaLinear(loss_type="l2")
    W = model.fit(X_scaled, lambda1=0.02)
    return W.T, time.time() - t0   # DAGMA returns W s.t. X = X*W + noise -> transpose to i->j


def dummy_fallback(X_scaled: np.ndarray, dataset_name: str) -> tuple:
    """Sparse lower-triangular random DAG — only for dry-run testing."""
    print(f"  [DUMMY fallback] {dataset_name} — install DrBO for real results.")
    rng = np.random.default_rng(args.seed)
    d = X_scaled.shape[1]
    adj = np.tril(
        rng.standard_normal((d, d)) * (rng.random((d, d)) > 0.6), -1
    )
    return adj, 0.0


def discover_dag(X_scaled: np.ndarray, task_type: str, dataset_name: str) -> tuple:
    if DRBO_AVAILABLE:
        return run_drbo(X_scaled, task_type, dataset_name)
    elif DAGMA_AVAILABLE:
        return run_dagma_fallback(X_scaled, dataset_name)
    else:
        return dummy_fallback(X_scaled, dataset_name)


# Graph saving

def save_graph(adj_weighted: np.ndarray, feat_names: list,
               dataset_name: str, method_tag: str = "drbo"):
    """Save weighted adjacency CSV, edge-list CSV, and PNG visualisation."""
    d = len(feat_names)
    tag = f"{dataset_name}_{method_tag}"

    # 1. Weighted adjacency matrix
    adj_df = pd.DataFrame(adj_weighted, index=feat_names, columns=feat_names)
    adj_path = os.path.join(OUT_DIR, f"{tag}_adj_matrix.csv")
    adj_df.to_csv(adj_path)

    # 2. Edge list with weights
    rows = []
    for i in range(d):
        for j in range(d):
            w = adj_weighted[i, j]
            if abs(w) > 1e-6:
                rows.append({
                    "source"     : feat_names[i],
                    "target_node": feat_names[j],
                    "weight"     : round(w, 6),
                })
    edges_df = pd.DataFrame(rows)
    edges_path = os.path.join(OUT_DIR, f"{tag}_graph_edges.csv")
    edges_df.to_csv(edges_path, index=False)

    # 3. Graph visualisation
    G = nx.DiGraph()
    G.add_nodes_from(feat_names)
    for _, row in edges_df.iterrows():
        G.add_edge(row["source"], row["target_node"], weight=row["weight"])

    fig, ax = plt.subplots(figsize=(max(8, d), max(6, d * 0.6)))
    pos = nx.spring_layout(G, seed=42, k=2.0)
    nx.draw_networkx(
        G, pos=pos, ax=ax,
        node_color="#2980b9", node_size=900,
        font_size=7, font_color="white",
        edge_color="gray", arrows=True, arrowsize=15, width=1.5,
    )
    edge_labels = {(u, v): f"{data['weight']:.2f}"
                   for u, v, data in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                 font_size=6, ax=ax)
    ax.set_title(f"DrBO Causal Graph — {dataset_name}  "
                 f"({len(edges_df)} edges)", fontsize=11)
    ax.axis("off")
    fig.tight_layout()
    img_path = os.path.join(GRAPH_DIR, f"{tag}_graph.png")
    fig.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"    adj matrix → {adj_path}")
    print(f"    edge list  → {edges_path}")
    print(f"    graph img  → {img_path}")
    return edges_df


# Causal shap - permutation sampling

def _causal_permutation(adj: np.ndarray, rng: np.random.Generator) -> list:
    """
    Draw one permutation that respects the causal (topological) ordering.
    Within each BFS generation the order is randomised.
    """
    d = adj.shape[0]
    G = nx.DiGraph()
    for i in range(d):
        for j in range(d):
            if abs(adj[i, j]) > 1e-6:
                G.add_edge(i, j)
    G.add_nodes_from(range(d))
    perm = []
    for layer in nx.topological_generations(G):
        layer_list = list(layer)
        rng.shuffle(layer_list)
        perm.extend(layer_list)
    return perm


def build_predictor(X: np.ndarray, y: np.ndarray, task_type: str):
    """
    Fit a simple sklearn model and return a callable predict(X) → 1-D array.
    For multiclass: predict probability of the most frequent class (class 0).
    For binary: predict P(class=1).
    For regression: predict raw value.
    """
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    if task_type == "regression":
        model = LinearRegression()
        model.fit(Xs, y)
        def predict(Z):
            return model.predict(scaler.transform(Z))

    elif task_type == "binary":
        model = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
        model.fit(Xs, y)
        def predict(Z):
            return model.predict_proba(scaler.transform(Z))[:, 1]

    else:  # multiclass
        model = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
        model.fit(Xs, y)
        # Use max predicted probability as scalar output for SHAP
        def predict(Z):
            return model.predict_proba(scaler.transform(Z)).max(axis=1)

    return predict


def causal_shap(
    X: np.ndarray,
    y: np.ndarray,
    adj_weighted: np.ndarray,
    task_type: str,
    n_samples: int = 200,
    M: int = 512,
    seed: int = 42,
) -> tuple:
    """
    Causal SHAP via causal permutation sampling.

    Returns:
        shap_vals  (n_samples × d) array of Shapley values
        eval_idx   (n_samples,)   indices of evaluated samples in X
    """
    rng = np.random.default_rng(seed)
    n, d = X.shape

    predict = build_predictor(X, y, task_type)

    eval_idx = rng.choice(n, size=min(n_samples, n), replace=False)
    X_eval   = X[eval_idx]
    shap_vals = np.zeros((len(eval_idx), d))

    for s_idx, x_eval in enumerate(X_eval):
        phi = np.zeros(d)
        for _ in range(M):
            perm   = _causal_permutation(adj_weighted, rng)
            bg_idx = rng.integers(n)
            x_curr = X[bg_idx].copy()
            prev_v = predict(x_curr.reshape(1, -1))[0]
            for feat in perm:
                x_curr[feat] = x_eval[feat]
                new_v = predict(x_curr.reshape(1, -1))[0]
                phi[feat] += (new_v - prev_v)
                prev_v = new_v
        shap_vals[s_idx] = phi / M

    return shap_vals, eval_idx


# Metrics

def compute_metrics(attribution: np.ndarray, gamma: np.ndarray) -> dict:
    a = attribution / (attribution.sum() + 1e-12)
    g = gamma       / (gamma.sum()        + 1e-12)
    pearson,  _ = pearsonr(a, g)
    spearman, _ = spearmanr(a, g)
    cos_sim     = 1.0 - cosine_dist(a, g)
    rmse        = float(np.sqrt(np.mean((a - g) ** 2)))
    return {"pearson": round(pearson,4), "spearman": round(spearman,4),
            "cosine_sim": round(cos_sim,4), "rmse": round(rmse,6)}


# Bar chart

def plot_bar_chart(mean_shap: np.ndarray, feat_names: list,
                   dataset_name: str, gamma: np.ndarray = None):
    """
    Mean |SHAP| bar chart.
    Style: tab10 colours, rotation=90 x-labels, no bar edges, dpi=600.
    """
    cmap   = plt.get_cmap("tab10")
    colors = [cmap(i % 10) for i in range(len(feat_names))]

    fig, ax = plt.subplots(figsize=(max(10, len(feat_names) * 0.75), 5))
    ax.bar(feat_names, mean_shap, color=colors, edgecolor="none")

    if gamma is not None and len(gamma) == len(feat_names):
        gamma_norm = gamma / (gamma.sum() + 1e-12) * mean_shap.sum()
        ax.plot(feat_names, gamma_norm, color="black", lw=1.5,
                ls="--", marker="o", ms=4, label="Ground Truth γ")
        ax.legend(fontsize=9)

    ax.set_xlabel("Feature", fontsize=11)
    ax.set_ylabel("Mean |Causal SHAP|", fontsize=11)
    ax.set_title(f"Causal SHAP (DrBO graph) — {dataset_name}", fontsize=12)
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names, rotation=90, ha="center", fontsize=8)
    fig.tight_layout()

    out = os.path.join(BARCHART_DIR, f"{dataset_name}_drbo_causal_shap.png")
    fig.savefig(out, dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"    bar chart  → {out}")


# Per-dataset pipeline

def run_dataset(loader_fn) -> dict | None:
    try:
        X, y, gamma, feat_names, task_type, dname = loader_fn()
    except FileNotFoundError as e:
        print(f"  [SKIP] {e}")
        return None

    n, d = X.shape
    print(f"\n{'='*60}")
    print(f"  {dname.upper()}  |  n={n}  d={d}  task={task_type}")
    print(f"{'='*60}")

    # Standardise for causal discovery (features only)
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 1. Causal discovery
    print("  [1/4] Causal discovery (DrBO) …")
    adj_weighted, t_disc = discover_dag(X_scaled, task_type, dname)

    # 2. Save graph (using original feature scale labels)
    print("  [2/4] Saving causal graph …")
    save_graph(adj_weighted, feat_names, dname)

    # 3. Causal SHAP (on original X, not scaled - predictor handles scaling)
    print("  [3/4] Computing Causal SHAP …")
    t1 = time.time()
    shap_vals, eval_idx = causal_shap(
        X, y, adj_weighted, task_type,
        n_samples=args.n_shap_samples, M=args.M, seed=args.seed,
    )
    t_shap = time.time() - t1
    mean_shap = np.mean(np.abs(shap_vals), axis=0)
    print(f"    done in {t_shap:.1f}s")

    # Save per-sample SHAP CSV
    shap_df = pd.DataFrame(shap_vals, columns=feat_names)
    shap_df.insert(0, "sample_idx", eval_idx)
    shap_path = os.path.join(OUT_DIR, f"{dname}_drbo_causal_shap_values.csv")
    shap_df.to_csv(shap_path, index=False)
    print(f"    shap csv   → {shap_path}")

    # 4. Metrics & bar chart
    print("  [4/4] Metrics & charts …")
    metrics_row = {
        "dataset"    : dname,
        "n_samples"  : n,
        "n_features" : d,
        "task_type"  : task_type,
        "n_edges"    : int((np.abs(adj_weighted) > 1e-6).sum()),
        "time_disc_s": round(t_disc, 2),
        "time_shap_s": round(t_shap, 2),
    }
    if gamma is not None and len(gamma) == d:
        m = compute_metrics(mean_shap, gamma)
        metrics_row.update(m)
        print(f"    metrics: {m}")
    else:
        metrics_row.update({"pearson": np.nan, "spearman": np.nan,
                            "cosine_sim": np.nan, "rmse": np.nan})

    pd.DataFrame([metrics_row]).to_csv(
        os.path.join(OUT_DIR, f"{dname}_drbo_metrics.csv"), index=False)

    plot_bar_chart(mean_shap, feat_names, dname, gamma)

    # Feature importance table (sorted)
    importance_df = pd.DataFrame({
        "feature"  : feat_names,
        "mean_abs_shap": mean_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    importance_df.to_csv(
        os.path.join(OUT_DIR, f"{dname}_drbo_feature_importance.csv"), index=False)
    print(f"\n  Feature importance ({dname}):")
    print(importance_df.to_string(index=False))

    return metrics_row


# Summary chart

def plot_summary(df: pd.DataFrame):
    metrics = ["pearson", "spearman", "cosine_sim", "rmse"]
    df_plot = df.dropna(subset=metrics)
    if df_plot.empty:
        return
    cmap = plt.get_cmap("tab10")
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, m in zip(axes, metrics):
        vals   = df_plot[m].values
        labels = df_plot["dataset"].values
        colors = [cmap(i % 10) for i in range(len(labels))]
        ax.bar(labels, vals, color=colors, edgecolor="none")
        ax.set_title(m, fontsize=12)
        ax.set_xticklabels(labels, rotation=90, fontsize=9)
        if m != "rmse":
            ax.set_ylim(-1, 1)
    fig.suptitle("Causal SHAP (DrBO) vs Ground Truth γ — All Datasets", fontsize=13)
    fig.tight_layout()
    out = os.path.join(BARCHART_DIR, "all_datasets_summary_metrics.png")
    fig.savefig(out, dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  Summary chart → {out}")


# Main

def main():
    print("\n" + "="*65)
    print("  Causal SHAP + DrBO  |  6 Datasets")
    print(f"  data_dir   = {DATA_DIR}")
    print(f"  n_evals    = {args.n_evals}")
    print(f"  n_shap_samples = {args.n_shap_samples}  M = {args.M}")
    print("="*65 + "\n")

    all_metrics = []
    for loader in DATASET_LOADERS:
        row = run_dataset(loader)
        if row:
            all_metrics.append(row)

    if all_metrics:
        summary = pd.DataFrame(all_metrics)
        summary_path = os.path.join(OUT_DIR, "all_datasets_summary.csv")
        summary.to_csv(summary_path, index=False)
        print(f"\n{'='*65}")
        print("FINAL SUMMARY")
        print(f"{'='*65}")
        print(summary.to_string(index=False))
        print(f"\nSaved → {summary_path}")
        plot_summary(summary)

    print("\nAll done.")


if __name__ == "__main__":
    main()

[OK] DrBO packages imported successfully in Jupyter environment.
[WARN] DrBO not found — will use DAGMA Linear fallback.
       Install: git clone https://github.com/baosws/DrBO.git
                pip install -r DrBO/requirements.txt
[OK] DAGMA found (fallback ready).

  Causal SHAP + DrBO  |  6 Datasets
  data_dir   = ../datasets
  n_evals    = 20000
  n_shap_samples = 200  M = 512


  YIELD  |  n=1000  d=4  task=multiclass
  [1/4] Causal discovery (DrBO) …
  [DAGMA fallback] yield


  [2/4] Saving causal graph …
    adj matrix → results_drbo/yield_drbo_adj_matrix.csv
    edge list  → results_drbo/yield_drbo_graph_edges.csv
    graph img  → results_drbo/graphs/yield_drbo_graph.png
  [3/4] Computing Causal SHAP …
    done in 86.9s
    shap csv   → results_drbo/yield_drbo_causal_shap_values.csv
  [4/4] Metrics & charts …
    bar chart  → results_drbo/bar_charts/yield_drbo_causal_shap.png

  Feature importance (yield):
      feature  mean_abs_shap
   Irrigation       0.068943
Precipitation       0.039232
     Nitrogen       0.031218
    Herbicide       0.011111

  SOIL  |  n=5000  d=7  task=multiclass
  [1/4] Causal discovery (DrBO) …
  [DAGMA fallback] soil


  [2/4] Saving causal graph …
    adj matrix → results_drbo/soil_drbo_adj_matrix.csv
    edge list  → results_drbo/soil_drbo_graph_edges.csv
    graph img  → results_drbo/graphs/soil_drbo_graph.png
  [3/4] Computing Causal SHAP …
    done in 135.5s
    shap csv   → results_drbo/soil_drbo_causal_shap_values.csv
  [4/4] Metrics & charts …
    bar chart  → results_drbo/bar_charts/soil_drbo_causal_shap.png

  Feature importance (soil):
                feature  mean_abs_shap
                     pH       0.095009
               Nitrogen       0.052897
              Potassium       0.033689
         Organic Carbon       0.013361
Electrical Conductivity       0.012504
             Phosphorus       0.006857
                   Zinc       0.003681

  CV_RISK  |  n=1000  d=4  task=regression
  [1/4] Causal discovery (DrBO) …
  [DAGMA fallback] cv_risk


  [2/4] Saving causal graph …
    adj matrix → results_drbo/cv_risk_drbo_adj_matrix.csv
    edge list  → results_drbo/cv_risk_drbo_graph_edges.csv
    graph img  → results_drbo/graphs/cv_risk_drbo_graph.png
  [3/4] Computing Causal SHAP …
    done in 63.0s
    shap csv   → results_drbo/cv_risk_drbo_causal_shap_values.csv
  [4/4] Metrics & charts …
    bar chart  → results_drbo/bar_charts/cv_risk_drbo_causal_shap.png

  Feature importance (cv_risk):
       feature  mean_abs_shap
           bmi       1.917071
    diet_score       0.046066
sleep_duration       0.042174
family_history       0.041539

  LUNG_CANCER  |  n=1000  d=3  task=regression
  [1/4] Causal discovery (DrBO) …
  [DAGMA fallback] lung_cancer


  [2/4] Saving causal graph …
    adj matrix → results_drbo/lung_cancer_drbo_adj_matrix.csv
    edge list  → results_drbo/lung_cancer_drbo_graph_edges.csv
    graph img  → results_drbo/graphs/lung_cancer_drbo_graph.png
  [3/4] Computing Causal SHAP …
    done in 50.7s
    shap csv   → results_drbo/lung_cancer_drbo_causal_shap_values.csv
  [4/4] Metrics & charts …
    bar chart  → results_drbo/bar_charts/lung_cancer_drbo_causal_shap.png

  Feature importance (lung_cancer):
     feature  mean_abs_shap
     smoking       2.241228
      stress       1.367191
drink_coffee       0.094795

  WINE  |  n=177  d=13  task=multiclass
  [1/4] Causal discovery (DrBO) …
  [DAGMA fallback] wine


  [2/4] Saving causal graph …
    adj matrix → results_drbo/wine_drbo_adj_matrix.csv
    edge list  → results_drbo/wine_drbo_graph_edges.csv
    graph img  → results_drbo/graphs/wine_drbo_graph.png
  [3/4] Computing Causal SHAP …
    done in 213.2s
    shap csv   → results_drbo/wine_drbo_causal_shap_values.csv
  [4/4] Metrics & charts …
    bar chart  → results_drbo/bar_charts/wine_drbo_causal_shap.png

  Feature importance (wine):
                     feature  mean_abs_shap
                     Proline       0.058132
                     Alcohol       0.042272
             Color intensity       0.040919
                  Flavanoids       0.040591
OD280/OD315 of diluted wines       0.035407
                         Hue       0.025065
                         Ash       0.020220
           Alcalinity of ash       0.017183
                  Malic acid       0.009335
             Proanthocyanins       0.006717
               Total phenols       0.003423
        Nonflavanoid phenols       0

  [2/4] Saving causal graph …
    adj matrix → results_drbo/compas_drbo_adj_matrix.csv
    edge list  → results_drbo/compas_drbo_graph_edges.csv
    graph img  → results_drbo/graphs/compas_drbo_graph.png
  [3/4] Computing Causal SHAP …
    done in 289.9s
    shap csv   → results_drbo/compas_drbo_causal_shap_values.csv
  [4/4] Metrics & charts …
    bar chart  → results_drbo/bar_charts/compas_drbo_causal_shap.png

  Feature importance (compas):
                feature  mean_abs_shap
           priors_count       0.108122
                    age       0.096190
        juv_other_count       0.011335
                  sex_1       0.011031
                  sex_0       0.011017
      c_charge_degree_F       0.008969
      c_charge_degree_M       0.008962
        age_cat_25-45_0       0.008100
        age_cat_25-45_1       0.008099
age_cat_Greaterthan45_1       0.006111
age_cat_Greaterthan45_0       0.006031
         juv_misd_count       0.001843
       race_Caucasian_1       0.001508
race_A

In [4]:
import os, sys

print("Active Directory:", os.getcwd())

# Check if the folder is visible
drbo_folder_path = os.path.join(os.getcwd(), "DrBO")
print("Does 'DrBO' folder exist here?:", os.path.isdir(drbo_folder_path))

# Force append it to make absolutely sure Python sees it
if os.path.isdir(drbo_folder_path):
    if drbo_folder_path not in sys.path:
        sys.path.insert(0, drbo_folder_path)

    # Try an explicit import to test it
    try:
        from drbo import DrBO
        print("Success! DrBO is now fully accessible to this notebook session.")
    except Exception as e:
        print("Folder found, but packages failed to initialize. Error:", e)

Active Directory: causal_bayesian_op
Does 'DrBO' folder exist here?: True
Success! DrBO is now fully accessible to this notebook session.


In [12]:
"""
causal_shap_drbo.py  —  Causal SHAP + DrBO  |  6 Datasets
===========================================================
Two-gamma architecture (confirmed from notebook test.ipynb):

  γ_est  : estimated gamma, derived from OLS on [features + target]
            → used AS the weight inside Causal SHAP: φ[i] += γ_est[i]*(v(S∪i)-v(S))
            → shown as feature→target edge weights on the causal graph

  γ_gt   : ground-truth gamma from each dataset's SEM/DGP
            → used ONLY to compute evaluation metrics (Pearson/Spearman/Cosine/RMSE)
            → shown as dashed overlay on bar charts

For DrBO:
  - DrBO runs on X_features only → discovers inter-feature DAG
  - OLS on [X_features, y] → W column for y gives γ_est per feature
  - γ_est is then used inside Causal SHAP AND as graph edge weights

Setup:
  git clone https://github.com/baosws/DrBO.git
  pip install -r DrBO/requirements.txt
  pip install scikit-learn pandas numpy scipy networkx matplotlib

Usage:
  python causal_shap_drbo.py [--data_dir /path/to/datasets]
"""

import os, sys, time, warnings, argparse
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.multiclass import OneVsRestClassifier

warnings.filterwarnings("ignore")

parser = argparse.ArgumentParser()
parser.add_argument("--data_dir",       default="../../../datasets")
parser.add_argument("--n_evals",        type=int, default=20000)
parser.add_argument("--n_shap_samples", type=int, default=200)
parser.add_argument("--M",              type=int, default=512)
parser.add_argument("--seed",           type=int, default=42)
args, _ = parser.parse_known_args()

DATA_DIR     = args.data_dir
OUT_DIR      = "results_drbo"
BARCHART_DIR = os.path.join(OUT_DIR, "bar_charts")
GRAPH_DIR    = os.path.join(OUT_DIR, "graphs")
for _d in [OUT_DIR, BARCHART_DIR, GRAPH_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── DrBO import ───────────────────────────────────────────────────────────────
DRBO_ROOT = os.path.join(os.getcwd(), "DrBO")
if os.path.isdir(DRBO_ROOT):
    sys.path.insert(0, DRBO_ROOT)
try:
    from drbo import DrBO
    from drbo.pruners import prune_linear
    DRBO_AVAILABLE = True
    print("[OK] DrBO loaded.")
except ImportError:
    DRBO_AVAILABLE = False
    print("[WARN] DrBO not found — dummy fallback.\n"
          "  git clone https://github.com/baosws/DrBO.git\n"
          "  pip install -r DrBO/requirements.txt")

try:
    from dagma.linear import DagmaLinear
    DAGMA_AVAILABLE = True
except ImportError:
    DAGMA_AVAILABLE = False


#  GROUND-TRUTH GAMMA (γ_gt) - used ONLY for metrics, never inside SHAP

def gt_gamma_soil(gt_path: str) -> np.ndarray:
    """Normalised mean |net SEM contribution|. Feature order: OC,N,P,K,PH,ZN,EC"""
    gt  = pd.read_csv(gt_path)
    net = np.array([
        np.abs(gt['OC_base'] - gt['OC_penalty']).mean(),
        np.abs(gt['N_base']  - gt['N_penalty']).mean(),
        np.abs(gt['P_base']  - gt['P_penalty']).mean(),
        np.abs(gt['K_base']  - gt['K_penalty']).mean(),
        gt['PH_penalty'].abs().mean(),
        np.abs(gt['ZN_base'] - gt['ZN_penalty']).mean(),
        gt['EC_penalty'].abs().mean(),
    ])
    return net / net.sum()


def gt_gamma_yield() -> np.ndarray:
    """Screenshot ground truth. NPZ order: Nitrogen, Irrigation, Precipitation, Herbicide"""
    g = np.array([0.296, 0.397, 0.220, 0.088])
    return g / g.sum()


def gt_gamma_cv_risk() -> np.ndarray:
    """Variance decomposition of DGP. Order: diet_score, sleep_duration, family_history, bmi"""
    var_diet = (9.0**2) / 12.0
    var_sleep = 4.0
    var_bmi   = 0.16*var_diet + 0.25*var_sleep + 1.0
    var_cv    = 2.25*var_bmi  + 3.0
    g = np.array([0.0, 0.0, 0.0, 2.25*var_bmi/var_cv])
    return g / g.sum()


def gt_gamma_lung_cancer() -> np.ndarray:
    """Variance decomposition of DGP. Order: smoking, stress, drink_coffee"""
    var_s, var_st, var_eps = 2.0, 2.0, 3.0
    var_y = 4.0*var_s + 1.44*var_st + var_eps
    raw   = np.array([4.0*var_s/var_y, 1.44*var_st/var_y, 0.0])
    tot   = raw[:2].sum()
    return raw / tot


#  ESTIMATED GAMMA (γ_est) - derived from OLS on [features + target]
#  This is used INSIDE Causal SHAP and as graph edge weights

def estimate_gamma(X: np.ndarray, y: np.ndarray, task: str) -> np.ndarray:
    """
    Run OLS/logistic regression on [X_features | y] to estimate
    the causal effect of each feature on the target.

    For regression:  OLS coefficients
    For multiclass:  mean |coef| across one-vs-rest classifiers

    Returns normalised γ_est array (shape d,), all non-negative, sums to 1.
    """
    d = X.shape[1]
    Xs = StandardScaler().fit_transform(X)

    if task == "regression":
        reg   = LinearRegression().fit(Xs, y)
        coefs = np.abs(reg.coef_)
    elif task == "binary":
        reg   = LogisticRegression(max_iter=1000, C=1.0).fit(Xs, y)
        coefs = np.abs(reg.coef_[0])
    else:  # multiclass
        reg   = LogisticRegression(max_iter=1000, C=1.0).fit(Xs, y)
        coefs = np.abs(reg.coef_).mean(axis=0)   # mean over classes

    coefs = np.maximum(coefs, 0.0)
    total = coefs.sum()
    if total < 1e-12:
        return np.ones(d) / d
    return coefs / total


# Dataset loaders
#  Returns: X, y, gamma_gt, feat_names, task, dname, target_name

def _npz(fname):
    d = np.load(os.path.join(DATA_DIR, fname), allow_pickle=True)
    X = np.concatenate([d['X_train'], d['X_test']], axis=0)
    y = np.concatenate([d['y_train'], d['y_test']], axis=0)
    return X, y, [str(f) for f in d['feature_names']]


def load_soil():
    X, y, feat_names = _npz("soil_spi_preprocessed.npz")
    gt_path = os.path.join(DATA_DIR, "ground_truth.csv")
    gamma_gt = gt_gamma_soil(gt_path) if os.path.exists(gt_path) else None
    return X, y, gamma_gt, feat_names, "multiclass", "soil", "SPI Class"


def load_yield():
    X, y, feat_names = _npz("yield_preprocessed.npz")
    return X, y, gt_gamma_yield(), feat_names, "multiclass", "yield", "Crop Yield"


def load_cv_risk():
    csv  = os.path.join(DATA_DIR, "cv_risk.csv")
    feat = ['diet_score', 'sleep_duration', 'family_history', 'bmi']
    tgt  = 'cv_risk'
    df   = pd.read_csv(csv).drop(columns=[c for c in
           pd.read_csv(csv, nrows=0).columns if 'Unnamed' in c])
    X, y = df[feat].values.astype(float), df[tgt].values.astype(float)
    return X, y, gt_gamma_cv_risk(), feat, "regression", "cv_risk", "CV Risk"


def load_lung_cancer():
    csv  = os.path.join(DATA_DIR, "lung_cancer_risk.csv")
    feat = ['smoking', 'stress', 'drink_coffee']
    tgt  = 'lung_cancer_risk'
    df   = pd.read_csv(csv).drop(columns=[c for c in
           pd.read_csv(csv, nrows=0).columns if 'Unnamed' in c])
    X, y = df[feat].values.astype(float), df[tgt].values.astype(float)
    return X, y, gt_gamma_lung_cancer(), feat, "regression", "lung_cancer", "Lung Cancer Risk"


def load_wine():
    X, y, feat_names = _npz("wine_preprocessed.npz")
    return X, y, None, feat_names, "multiclass", "wine", "Wine Class"


def load_compas():
    X, y, feat_names = _npz("compas_preprocessed.npz")
    return X, y, None, feat_names, "binary", "compas", "Recidivism"


LOADERS = [load_soil, load_yield, load_cv_risk,
           load_lung_cancer, load_wine, load_compas]


#  CAUSAL DISCOVERY  (DrBO - features only)

def run_drbo(X_sc, dname):
    t0  = time.time()
    print(f"    Running DrBO ({args.n_evals} evals) — this may take several minutes …")
    ret = DrBO(X=X_sc, score_method='BIC',
               score_params={'noise_var': 'nv', 'reg': 'linear'},
               max_evals=args.n_evals, batch_size=64, dag_rank=8,
               n_cands=100_000, n_grads=10, lr=0.1, hidden_size=64,
               dropout=0.1, n_replay=1024, normalize=False,
               pruner=prune_linear, device="cpu", verbose=False,
               random_state=args.seed)
    elapsed = time.time() - t0
    raw = ret['raw']   # (d×d) binary DAG, raw[i,j]=1 -> i->j
    d   = X_sc.shape[1]
    # Recover signed inter-feature edge weights via OLS per node
    adj = np.zeros((d, d))
    for j in range(d):
        parents = np.where(raw[:, j] == 1)[0]
        if len(parents):
            r = Ridge(alpha=1e-3).fit(X_sc[:, parents], X_sc[:, j])
            for ki, p in enumerate(parents):
                adj[p, j] = float(r.coef_[ki])
    print(f"  [DrBO] {dname}: {int(raw.sum())} edges, {elapsed:.1f}s")
    return adj, elapsed


def run_dagma(X_sc, dname):
    print(f"  [DAGMA fallback] {dname}")
    t0 = time.time()
    W  = DagmaLinear(loss_type="l2").fit(X_sc, lambda1=0.02)
    return W.T, time.time() - t0


def dummy_dag(X_sc, dname):
    print(f"  [DUMMY fallback] {dname}")
    rng = np.random.default_rng(args.seed)
    d   = X_sc.shape[1]
    return np.tril(rng.standard_normal((d, d))*(rng.random((d,d))>0.65), -1), 0.0


def discover(X_sc, dname):
    if DRBO_AVAILABLE:  return run_drbo(X_sc, dname)
    if DAGMA_AVAILABLE: return run_dagma(X_sc, dname)
    return dummy_dag(X_sc, dname)


# Causal shap
#  γ_est is the weight inside the Shapley sum:

def _causal_perm(adj, rng):
    d = adj.shape[0]
    G = nx.DiGraph()
    G.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if abs(adj[i,j]) > 1e-6:
                G.add_edge(i, j)
    perm = []
    for layer in nx.topological_generations(G):
        lst = list(layer); rng.shuffle(lst); perm.extend(lst)
    return perm


def _build_predictor(X, y, task):
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    if task == "regression":
        m = LinearRegression().fit(Xs, y)
        return lambda Z: m.predict(sc.transform(Z))
    elif task == "binary":
        m = LogisticRegression(max_iter=1000, C=1.0).fit(Xs, y)
        return lambda Z: m.predict_proba(sc.transform(Z))[:,1]
    else:
        m = LogisticRegression(max_iter=1000, C=1.0).fit(Xs, y)
        return lambda Z: m.predict_proba(sc.transform(Z)).max(axis=1)


def causal_shap(X, y, adj, gamma_est, task):
    """
    Causal SHAP with γ_est as the within-Shapley weighting factor.

    φ[i] = (1/M) Σ_perms  γ_est[i] * (v(S∪{i}) - v(S))
    """
    rng     = np.random.default_rng(args.seed)
    n, d    = X.shape
    predict = _build_predictor(X, y, task)
    idx     = rng.choice(n, size=min(args.n_shap_samples, n), replace=False)
    phi     = np.zeros((len(idx), d))

    for si, x_ev in enumerate(X[idx]):
        acc = np.zeros(d)
        for _ in range(args.M):
            perm   = _causal_perm(adj, rng)
            x_curr = X[rng.integers(n)].copy()
            prev   = predict(x_curr.reshape(1,-1))[0]
            for f in perm:
                x_curr[f] = x_ev[f]
                nxt = predict(x_curr.reshape(1,-1))[0]
                # Weight each marginal contribution by γ_est[f]
                acc[f] += gamma_est[f] * (nxt - prev)
                prev = nxt
        phi[si] = acc / args.M
    return phi, idx


#  METRICS  - compare SHAP attributions against γ_gt

def compute_metrics(mean_shap: np.ndarray, gamma_gt: np.ndarray) -> dict:
    """Pearson/Spearman/Cosine/RMSE of normalised SHAP vs ground-truth gamma."""
    a = mean_shap / (mean_shap.sum() + 1e-12)
    g = gamma_gt  / (gamma_gt.sum()  + 1e-12)
    rp, _ = pearsonr(a, g);   rs, _ = spearmanr(a, g)
    cos   = 1.0 - cosine_dist(a, g)
    rmse  = float(np.sqrt(np.mean((a - g)**2)))
    return dict(pearson=round(rp,4), spearman=round(rs,4),
                cosine_sim=round(cos,4), rmse=round(rmse,6))


# Graph visualisation
#  Feature->target edge weight = γ_est (estimated causal effect from OLS)

POS_COL    = '#1B8C7A'
NEG_COL    = '#C0392B'
EST_COL    = '#8E24AA'    # purple - γ_est edges
TARGET_COL = '#6A1B9A'
BLUES      = ['#AED6F1','#5DADE2','#2E86C1','#1B5E87','#0D3349']


def _wrap(name, maxlen=12):
    parts = name.split()
    if len(name) <= maxlen or len(parts) == 1:
        return name
    mid = len(parts) // 2
    return ' '.join(parts[:mid]) + '\n' + ' '.join(parts[mid:])


def draw_causal_graph(adj_feat: np.ndarray, feat_names: list,
                      gamma_est: np.ndarray,
                      target_name: str, title: str, out_path: str):
    """
    Graph with two edge types:
      Solid teal/coral  = DrBO inter-feature edges (structural)
      Dashed purple     = feature→target, weight = γ_est (OLS-estimated causal effect)
    """
    d = len(feat_names)
    G_feat = nx.DiGraph()
    G_feat.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if abs(adj_feat[i, j]) > 1e-6:
                G_feat.add_edge(i, j, weight=float(adj_feat[i, j]))

    try:
        layers = list(nx.topological_generations(G_feat))
    except nx.NetworkXUnfeasible:
        layers = [list(range(d))]

    pos = {}
    n_layers = len(layers)
    for li, layer in enumerate(layers):
        layer = sorted(layer)
        n = len(layer)
        y = 1.0 - li * (0.80 / max(n_layers-1, 1))
        for ki, node in enumerate(layer):
            pos[node] = ((ki+1)/(n+1), y)
    TARGET = d
    pos[TARGET] = (0.5, -0.08)

    figw = max(12, d * 1.3)
    figh = max(10, (n_layers + 2) * 1.8)
    fig, ax = plt.subplots(figsize=(figw, figh))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')

    # ── Inter-feature edges (DrBO, solid) ────────────────────────────────────
    feat_labels = []
    for src, dst, data in G_feat.edges(data=True):
        w   = data['weight']
        col = POS_COL if w > 0 else NEG_COL
        lw  = 0.9 + min(abs(w)*2.5, 4.0)
        xi, yi = pos[src]; xj, yj = pos[dst]
        dy_abs = abs(yi-yj)
        rad    = 0.0 if dy_abs > 0.35 else (0.25 if dy_abs < 0.05 else 0.12)
        ax.annotate("", xy=(xj,yj), xytext=(xi,yi),
            arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, alpha=0.72,
                            connectionstyle=f"arc3,rad={rad}",
                            shrinkA=23, shrinkB=23))
        t = 0.42; mx = xi+t*(xj-xi); my = yi+t*(yj-yi)
        dx, dy = xj-xi, yj-yi; norm = max(np.hypot(dx,dy), 1e-6)
        side = 1 if (src+dst)%2==0 else -1
        ox = side*(-dy/norm)*0.04; oy = side*(dx/norm)*0.04
        feat_labels.append((mx+ox, my+oy, f"{w:+.2f}", 'white', '#cccccc'))

    # ── Feature->target edges (γ_est, dashed purple) ──────────────────────────
    feat_sorted = sorted(range(d), key=lambda i: pos[i][0])
    tx, ty = pos[TARGET]
    tgt_labels = []
    for rank, fi in enumerate(feat_sorted):
        w = float(gamma_est[fi])
        if w < 1e-6:
            continue
        xi, yi = pos[fi]
        lw = 0.8 + min(w * 30, 3.5)
        spread = min(0.28, d * 0.04)
        arr_x  = tx + spread * (rank/(max(d-1, 1)) - 0.5)
        ax.annotate("", xy=(arr_x, ty+0.065), xytext=(xi, yi),
            arrowprops=dict(arrowstyle="-|>", color=EST_COL, lw=lw,
                            alpha=0.60, linestyle='dashed',
                            connectionstyle="arc3,rad=0.0",
                            shrinkA=23, shrinkB=26))
        t = 0.55; mx = xi+t*(arr_x-xi); my = yi+t*(ty+0.065-yi)
        dx, dy = arr_x-xi, ty+0.065-yi; norm = max(np.hypot(dx,dy), 1e-6)
        side = 1 if fi%2==0 else -1
        ox = side*(-dy/norm)*0.035; oy = side*(dx/norm)*0.035
        tgt_labels.append((mx+ox, my+oy, f"{w:.3f}"))

    for lx, ly, lab, fc, ec in feat_labels:
        ax.text(lx, ly, lab, ha='center', va='center', fontsize=7.5,
                color='#111111', fontweight='medium', zorder=8,
                bbox=dict(boxstyle='round,pad=0.18', fc=fc, ec=ec, alpha=0.90, lw=0.5))
    for lx, ly, lab in tgt_labels:
        ax.text(lx, ly, lab, ha='center', va='center', fontsize=7.5,
                color='#4A148C', fontweight='semibold', zorder=10,
                bbox=dict(boxstyle='round,pad=0.15', fc='#F3E5F5',
                          ec='#CE93D8', alpha=0.93, lw=0.5))

    # ── Nodes ─────────────────────────────────────────────────────────────────
    in_deg = dict(G_feat.in_degree())
    for i in range(d):
        x, y  = pos[i]
        col   = BLUES[min(in_deg.get(i, 0), len(BLUES)-1)]
        label = _wrap(feat_names[i])
        lines = label.count('\n')+1
        bw    = max(0.10, len(max(label.split('\n'), key=len))*0.013)
        bh    = 0.055+(lines-1)*0.033
        ax.add_patch(mpatches.FancyBboxPatch(
            (x-bw, y-bh/2), bw*2, bh, boxstyle="round,pad=0.015",
            facecolor=col, edgecolor='white', linewidth=2, zorder=5))
        ax.text(x, y, label, ha='center', va='center', fontsize=8.5,
                color='white', fontweight='bold', zorder=6,
                multialignment='center', linespacing=1.25)

    tbw = max(0.14, len(target_name)*0.013)
    ax.add_patch(mpatches.FancyBboxPatch(
        (tx-tbw, ty-0.065), tbw*2, 0.13, boxstyle="round,pad=0.018",
        facecolor=TARGET_COL, edgecolor='white', linewidth=2.5, zorder=5))
    ax.text(tx, ty, target_name, ha='center', va='center',
            fontsize=10, color='white', fontweight='bold', zorder=6)

    handles = [
        plt.Line2D([0],[0], color=POS_COL,   lw=2,
                   label='Positive inter-feature effect (DrBO)'),
        plt.Line2D([0],[0], color=NEG_COL,   lw=2,
                   label='Negative inter-feature effect (DrBO)'),
        plt.Line2D([0],[0], color=EST_COL,   lw=2, ls='--',
                   label='Feature → target  (γ_est, OLS-estimated causal effect)'),
        mpatches.Patch(facecolor=BLUES[3],   label='High in-degree feature'),
        mpatches.Patch(facecolor=BLUES[0],   label='Low in-degree feature'),
        mpatches.Patch(facecolor=TARGET_COL, label='Target variable'),
    ]
    ax.legend(handles=handles, loc='lower left', fontsize=8.5,
              framealpha=0.95, edgecolor='#cccccc', fancybox=True, ncol=2)

    ax.set_title(title, fontsize=14, fontweight='bold', pad=16, color='#111111')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.28, 1.25)
    ax.axis('off')
    fig.tight_layout(pad=2.2)
    fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"    graph img  → {out_path}")


def save_graph_files(adj, feat_names, gamma_est, target_name, dname):
    tag = f"{dname}_drbo"
    d   = len(feat_names)

    # Weighted adjacency matrix (inter-feature only)
    pd.DataFrame(adj, index=feat_names, columns=feat_names).to_csv(
        os.path.join(OUT_DIR, f"{tag}_adj_matrix.csv"))

    # Edge list: inter-feature + feature->target (γ_est)
    rows = []
    for i in range(d):
        for j in range(d):
            if abs(adj[i,j]) > 1e-6:
                rows.append({"source": feat_names[i], "target_node": feat_names[j],
                             "weight": round(adj[i,j],6),
                             "edge_type": "inter_feature",
                             "weight_meaning": "DrBO OLS coefficient"})
    for i, f in enumerate(feat_names):
        rows.append({"source": f, "target_node": target_name,
                     "weight": round(float(gamma_est[i]),6),
                     "edge_type": "feature_to_target",
                     "weight_meaning": "gamma_est (OLS-estimated causal effect)"})
    pd.DataFrame(rows).to_csv(
        os.path.join(OUT_DIR, f"{tag}_graph_edges.csv"), index=False)

    draw_causal_graph(
        adj, feat_names, gamma_est, target_name,
        title=f"DrBO Causal Graph — {target_name}",
        out_path=os.path.join(GRAPH_DIR, f"{tag}_graph.png"))


#  BAR CHART - mean |SHAP| bars with γ_gt overlay

def plot_bar(mean_shap, feat_names, dname, gamma_gt=None):
    cmap   = plt.get_cmap("tab10")
    colors = [cmap(i%10) for i in range(len(feat_names))]
    fig, ax = plt.subplots(figsize=(max(10, len(feat_names)*0.8), 5))
    ax.bar(feat_names, mean_shap, color=colors, edgecolor='none')
    if gamma_gt is not None and len(gamma_gt) == len(feat_names):
        gn = gamma_gt/(gamma_gt.sum()+1e-12)*mean_shap.sum()
        ax.plot(feat_names, gn, color='black', lw=1.8, ls='--',
                marker='o', ms=5, label='Ground Truth γ (SEM)', zorder=5)
        ax.legend(fontsize=10)
    ax.set_xlabel("Feature", fontsize=11)
    ax.set_ylabel("Mean |Causal SHAP|", fontsize=11)
    ax.set_title(f"Causal SHAP (DrBO) — {dname}", fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names, rotation=90, ha='center', fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    fig.tight_layout()
    out = os.path.join(BARCHART_DIR, f"{dname}_drbo_causal_shap.png")
    fig.savefig(out, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"    bar chart  → {out}")


# Per-dataset pipeline

def run_dataset(loader):
    try:
        X, y, gamma_gt, feat_names, task, dname, target_name = loader()
    except FileNotFoundError as e:
        print(f"  [SKIP] {e}"); return None

    n, d = X.shape
    print(f"\n{'='*60}")
    print(f"  {dname.upper()}  n={n}  d={d}  task={task}")
    if gamma_gt is not None:
        print(f"  γ_gt : { {feat_names[i]: round(float(gamma_gt[i]),4) for i in range(d)} }")
    else:
        print(f"  γ_gt : None (Wine/COMPAS — no SEM ground truth)")
    print(f"{'='*60}")

    X_sc = StandardScaler().fit_transform(X)

    # ── 1. Causal discovery (DrBO, features only) ─────────────────────────────
    print("  [1/4] Causal discovery (DrBO, features only) …")
    adj, t_disc = discover(X_sc, dname)

    # ── 2. Estimate gamma via OLS on [features + target] ──────────────────────
    print("  [2/4] Estimating γ_est via OLS [features → target] …")
    gamma_est = estimate_gamma(X, y, task)
    print(f"    γ_est: { {feat_names[i]: round(float(gamma_est[i]),4) for i in range(d)} }")

    # ── 3. Causal SHAP (using γ_est as within-Shapley weights) ────────────────
    print("  [3/4] Causal SHAP (γ_est weighted) …")
    t1 = time.time()
    phi, eval_idx = causal_shap(X, y, adj, gamma_est, task)
    t_shap        = time.time() - t1
    mean_shap     = np.mean(np.abs(phi), axis=0)

    shap_df = pd.DataFrame(phi, columns=feat_names)
    shap_df.insert(0, "sample_idx", eval_idx)
    sp = os.path.join(OUT_DIR, f"{dname}_drbo_causal_shap_values.csv")
    shap_df.to_csv(sp, index=False)
    print(f"    done {t_shap:.1f}s → {sp}")

    # ── 4. Save graph (γ_est as feature->target edge weights) ──────────────────
    print("  [4/5] Saving graph …")
    save_graph_files(adj, feat_names, gamma_est, target_name, dname)

    # ── 5. Metrics vs γ_gt and bar charts ─────────────────────────────────────
    print("  [5/5] Metrics & charts …")
    row = dict(dataset=dname, n_samples=n, n_features=d, task=task,
               target=target_name,
               n_inter_edges=int((np.abs(adj)>1e-6).sum()),
               time_disc_s=round(t_disc,2), time_shap_s=round(t_shap,2))

    if gamma_gt is not None and len(gamma_gt) == d:
        m = compute_metrics(mean_shap, gamma_gt)
        row.update(m)
        print(f"    SHAP vs γ_gt: {m}")
    else:
        row.update(pearson=np.nan, spearman=np.nan, cosine_sim=np.nan, rmse=np.nan)
        print("    No γ_gt → metrics = NaN (correct for Wine/COMPAS)")

    pd.DataFrame([row]).to_csv(
        os.path.join(OUT_DIR, f"{dname}_drbo_metrics.csv"), index=False)

    plot_bar(mean_shap, feat_names, dname, gamma_gt)

    # Feature importance table
    imp = pd.DataFrame({
        "feature"   : feat_names,
        "mean_abs_shap": mean_shap,
        "gamma_est" : gamma_est,
        "gamma_gt"  : gamma_gt if gamma_gt is not None else [np.nan]*d,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    imp.to_csv(os.path.join(OUT_DIR, f"{dname}_drbo_feature_importance.csv"), index=False)
    print(f"\n  Feature importance ({dname}):\n{imp.to_string(index=False)}")
    return row


# Summary

def plot_summary(df):
    ms  = ["pearson","spearman","cosine_sim","rmse"]
    dfp = df.dropna(subset=ms)
    if dfp.empty: return
    cmap = plt.get_cmap("tab10")
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, m in zip(axes, ms):
        v = dfp[m].values; labels = dfp["dataset"].values
        ax.bar(labels, v, color=[cmap(i%10) for i in range(len(labels))], edgecolor='none')
        ax.set_title(m, fontsize=12, fontweight='bold')
        ax.set_xticklabels(labels, rotation=90, fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        if m != "rmse": ax.set_ylim(-1, 1)
    fig.suptitle("Causal SHAP (DrBO, γ_est-weighted) vs Ground Truth γ_gt",
                 fontsize=13, fontweight='bold')
    fig.tight_layout()
    out = os.path.join(BARCHART_DIR, "all_datasets_summary_metrics.png")
    fig.savefig(out, dpi=600, bbox_inches='tight'); plt.close()
    print(f"\n  Summary chart → {out}")


def main():
    print("\n"+"="*65)
    print("  Causal SHAP + DrBO  |  6 Datasets")
    print(f"  data_dir={DATA_DIR}  n_evals={args.n_evals}")
    print(f"  n_shap_samples={args.n_shap_samples}  M={args.M}")
    print("="*65)
    rows = [r for loader in LOADERS if (r := run_dataset(loader)) is not None]
    if rows:
        summary = pd.DataFrame(rows)
        sp = os.path.join(OUT_DIR, "all_datasets_summary.csv")
        summary.to_csv(sp, index=False)
        print(f"\n{'='*65}\nSUMMARY\n{'='*65}")
        print(summary.to_string(index=False))
        plot_summary(summary)
    print("\nAll done.")

if __name__ == "__main__":
    main()

[OK] DrBO loaded.

  Causal SHAP + DrBO  |  6 Datasets
  data_dir=../datasets  n_evals=20000
  n_shap_samples=200  M=512

  SOIL  n=5000  d=7  task=multiclass
  γ_gt : {'Organic Carbon': 0.0622, 'Nitrogen': 0.4012, 'Phosphorus': 0.0762, 'Potassium': 0.2588, 'pH': 0.0886, 'Zinc': 0.0147, 'Electrical Conductivity': 0.0984}
  [1/4] Causal discovery (DrBO, features only) …
    Running DrBO (20000 evals) — this may take several minutes …
  [DrBO] soil: 10 edges, 978.8s
  [2/4] Estimating γ_est via OLS [features → target] …
    γ_est: {'Organic Carbon': 0.0891, 'Nitrogen': 0.2632, 'Phosphorus': 0.0675, 'Potassium': 0.2684, 'pH': 0.1694, 'Zinc': 0.0249, 'Electrical Conductivity': 0.1175}
  [3/4] Causal SHAP (γ_est weighted) …
    done 120.1s → results_drbo/soil_drbo_causal_shap_values.csv
  [4/5] Saving graph …
    graph img  → results_drbo/graphs/soil_drbo_graph.png
  [5/5] Metrics & charts …
    SHAP vs γ_gt: {'pearson': np.float64(0.6032), 'spearman': np.float64(0.6071), 'cosine_sim': np.f

In [13]:
"""
causal_shap_drbo.py  —  Causal SHAP + DrBO  |  6 Datasets
===========================================================
Two-gamma architecture:
  γ_est : OLS-estimated causal effect → used inside Causal SHAP + graph edges
  γ_gt  : SEM ground truth            → used only for metrics (soil/yield/cv_risk/lung_cancer)

For Wine and COMPAS (no γ_gt):
  Metrics computed against MC Shapley reference, plus faithfulness and robustness,
  exactly matching the metric suite from the COMPAS notebook.

COMPAS fix: drop perfectly collinear _0 dummy columns before DrBO
  (sex_0+sex_1=1, age_cat_*_0+_1=1, etc → condition number 1e18 → singular XtX in BIC)
  Kept features mirror the 'preferred' list from the COMPAS notebook.

Setup:
  git clone https://github.com/baosws/DrBO.git && pip install -r DrBO/requirements.txt
  pip install scikit-learn xgboost pandas numpy scipy networkx matplotlib

Usage:
  python causal_shap_drbo.py [--data_dir /path/to/datasets]
"""

import os, sys, time, warnings, argparse
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

parser = argparse.ArgumentParser()
parser.add_argument("--data_dir",       default="../../../datasets")
parser.add_argument("--n_evals",        type=int, default=20000)
parser.add_argument("--n_shap_samples", type=int, default=200)
parser.add_argument("--M",              type=int, default=512)
parser.add_argument("--seed",           type=int, default=42)
# MC Shapley params (for Wine / COMPAS)
parser.add_argument("--mc_k",          type=int, default=40)
parser.add_argument("--mc_bg_size",    type=int, default=120)
parser.add_argument("--mc_iters",      type=int, default=600)
parser.add_argument("--robust_k",      type=int, default=15)
parser.add_argument("--perturb_runs",  type=int, default=5)
parser.add_argument("--perturb_sigma", type=float, default=0.05)
args, _ = parser.parse_known_args()

DATA_DIR     = args.data_dir
OUT_DIR      = "results_drbo"
BARCHART_DIR = os.path.join(OUT_DIR, "bar_charts")
GRAPH_DIR    = os.path.join(OUT_DIR, "graphs")
for _d in [OUT_DIR, BARCHART_DIR, GRAPH_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── DrBO ──────────────────────────────────────────────────────────────────────
DRBO_ROOT = os.path.join(os.getcwd(), "DrBO")
if os.path.isdir(DRBO_ROOT):
    sys.path.insert(0, DRBO_ROOT)
try:
    from drbo import DrBO
    from drbo.pruners import prune_linear
    DRBO_AVAILABLE = True
    print("[OK] DrBO loaded.")
except ImportError:
    DRBO_AVAILABLE = False
    print("[WARN] DrBO not found — dummy fallback.")

try:
    from dagma.linear import DagmaLinear
    DAGMA_AVAILABLE = True
except ImportError:
    DAGMA_AVAILABLE = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


#  GROUND-TRUTH GAMMA (γ_gt) - only for soil/yield/cv_risk/lung_cancer

def gt_gamma_soil(gt_path):
    gt  = pd.read_csv(gt_path)
    net = np.array([
        np.abs(gt['OC_base'] - gt['OC_penalty']).mean(),
        np.abs(gt['N_base']  - gt['N_penalty']).mean(),
        np.abs(gt['P_base']  - gt['P_penalty']).mean(),
        np.abs(gt['K_base']  - gt['K_penalty']).mean(),
        gt['PH_penalty'].abs().mean(),
        np.abs(gt['ZN_base'] - gt['ZN_penalty']).mean(),
        gt['EC_penalty'].abs().mean(),
    ])
    return net / net.sum()


def gt_gamma_yield():
    g = np.array([0.296, 0.397, 0.220, 0.088])   # NPZ order: N,I,P,H
    return g / g.sum()


def gt_gamma_cv_risk():
    var_diet = (9.0**2)/12.0; var_sleep = 4.0
    var_bmi  = 0.16*var_diet + 0.25*var_sleep + 1.0
    var_cv   = 2.25*var_bmi  + 3.0
    g = np.array([0.0, 0.0, 0.0, 2.25*var_bmi/var_cv])
    return g / g.sum()


def gt_gamma_lung_cancer():
    var_s, var_st, var_eps = 2.0, 2.0, 3.0
    var_y = 4.0*var_s + 1.44*var_st + var_eps
    raw   = np.array([4.0*var_s/var_y, 1.44*var_st/var_y, 0.0])
    return raw / raw[:2].sum()


def _npz(fname):
    d = np.load(os.path.join(DATA_DIR, fname), allow_pickle=True)
    return (d['X_train'], d['X_test'], d['y_train'], d['y_test'],
            [str(f) for f in d['feature_names']])


def load_soil():
    Xtr,Xte,ytr,yte,fn = _npz("soil_spi_preprocessed.npz")
    gt_path = os.path.join(DATA_DIR, "ground_truth.csv")
    gamma   = gt_gamma_soil(gt_path) if os.path.exists(gt_path) else None
    X = np.concatenate([Xtr,Xte]); y = np.concatenate([ytr,yte])
    return X,X,y,y, gamma, fn, "multiclass","soil","SPI Class"


def load_yield():
    Xtr,Xte,ytr,yte,fn = _npz("yield_preprocessed.npz")
    X = np.concatenate([Xtr,Xte]); y = np.concatenate([ytr,yte])
    return X,X,y,y, gt_gamma_yield(), fn, "multiclass","yield","Crop Yield"


def load_cv_risk():
    csv  = os.path.join(DATA_DIR, "cv_risk.csv")
    feat = ['diet_score','sleep_duration','family_history','bmi']
    df   = pd.read_csv(csv).drop(columns=[c for c in
           pd.read_csv(csv,nrows=0).columns if 'Unnamed' in c])
    X = df[feat].values.astype(float); y = df['cv_risk'].values.astype(float)
    return X,X,y,y, gt_gamma_cv_risk(), feat, "regression","cv_risk","CV Risk"


def load_lung_cancer():
    csv  = os.path.join(DATA_DIR, "lung_cancer_risk.csv")
    feat = ['smoking','stress','drink_coffee']
    df   = pd.read_csv(csv).drop(columns=[c for c in
           pd.read_csv(csv,nrows=0).columns if 'Unnamed' in c])
    X = df[feat].values.astype(float); y = df['lung_cancer_risk'].values.astype(float)
    return X,X,y,y, gt_gamma_lung_cancer(), feat, "regression","lung_cancer","Lung Cancer Risk"


def load_wine():
    Xtr,Xte,ytr,yte,fn = _npz("wine_preprocessed.npz")
    return Xtr,Xte,ytr,yte, None, fn, "multiclass","wine","Wine Class"


def load_compas():
    Xtr,Xte,ytr,yte,fn = _npz("compas_preprocessed.npz")
    preferred = ["age","juv_fel_count","juv_misd_count","juv_other_count",
                 "priors_count","c_charge_degree_F","sex_1",
                 "age_cat_25-45_1","age_cat_Greaterthan45_1","race_African-American_1"]
    kept = [c for c in preferred if c in fn]
    idx  = [fn.index(c) for c in kept]
    return Xtr[:,idx],Xte[:,idx],ytr,yte, None, kept, "binary","compas","Recidivism"


LOADERS = [load_wine, load_compas]


#  ESTIMATED GAMMA (γ_est) - OLS/logistic on [features -> target]

def estimate_gamma(X, y, task):
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    if task == "regression":
        coefs = np.abs(LinearRegression().fit(Xs,y).coef_)
    elif task == "binary":
        coefs = np.abs(LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y).coef_[0])
    else:
        coefs = np.abs(LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y).coef_).mean(0)
    coefs = np.maximum(coefs, 0.0)
    return coefs / (coefs.sum() + 1e-12)


# Causal discovery

def run_drbo(X_sc, dname):
    t0 = time.time()
    print(f"    Running DrBO ({args.n_evals} evals) — this may take several minutes …")
    ret = DrBO(X=X_sc, score_method='BIC',
               score_params={'noise_var':'nv','reg':'linear'},
               max_evals=args.n_evals, batch_size=64, dag_rank=8,
               n_cands=100_000, n_grads=10, lr=0.1, hidden_size=64,
               dropout=0.1, n_replay=1024, normalize=False,
               pruner=prune_linear, device="cpu", verbose=False,
               random_state=args.seed)
    elapsed = time.time() - t0
    raw = ret['raw']; d = X_sc.shape[1]
    adj = np.zeros((d,d))
    for j in range(d):
        parents = np.where(raw[:,j]==1)[0]
        if len(parents):
            r = Ridge(alpha=1e-3).fit(X_sc[:,parents], X_sc[:,j])
            for ki,p in enumerate(parents):
                adj[p,j] = float(r.coef_[ki])
    print(f"  [DrBO] {dname}: {int(raw.sum())} edges, {elapsed:.1f}s")
    return adj, elapsed


def run_dagma(X_sc, dname):
    t0 = time.time()
    W  = DagmaLinear(loss_type="l2").fit(X_sc, lambda1=0.02)
    return W.T, time.time()-t0


def dummy_dag(X_sc, dname):
    rng = np.random.default_rng(args.seed); d = X_sc.shape[1]
    return np.tril(rng.standard_normal((d,d))*(rng.random((d,d))>0.65),-1), 0.0


def discover(X_sc, dname):
    if DRBO_AVAILABLE:  return run_drbo(X_sc, dname)
    if DAGMA_AVAILABLE: return run_dagma(X_sc, dname)
    return dummy_dag(X_sc, dname)


#  CAUSAL SHAP (γ_est-weighted permutation)

def _causal_perm(adj, rng):
    d = adj.shape[0]
    G = nx.DiGraph(); G.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if abs(adj[i,j])>1e-6: G.add_edge(i,j)
    perm = []
    for layer in nx.topological_generations(G):
        lst = list(layer); rng.shuffle(lst); perm.extend(lst)
    return perm


def _build_predictor(X, y, task):
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    if task=="regression":
        m = LinearRegression().fit(Xs,y)
        return lambda Z: m.predict(sc.transform(Z))
    elif task=="binary":
        m = LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y)
        return lambda Z: m.predict_proba(sc.transform(Z))[:,1]
    else:
        m = LogisticRegression(max_iter=1000,C=1.0).fit(Xs,y)
        return lambda Z: m.predict_proba(sc.transform(Z)).max(axis=1)


def causal_shap(X_train, X_eval, adj, gamma_est, task):
    rng     = np.random.default_rng(args.seed)
    n,d     = X_train.shape
    predict = _build_predictor(X_train, None if task in ("binary","multiclass")
                               else X_train[:,0], task)
    # rebuild predictor properly
    predict = _build_predictor(X_train,
                               np.zeros(n) if task=="regression" else np.zeros(n,dtype=int),
                               task)
    # We need y - pass X_train and y together
    return None  # placeholder - see run_dataset for correct call


def _causal_shap_full(X_train, y_train, X_eval, adj, gamma_est, task):
    rng     = np.random.default_rng(args.seed)
    n_tr    = X_train.shape[0]; d = X_train.shape[1]
    predict = _build_predictor(X_train, y_train, task)
    n_eval  = min(args.n_shap_samples, len(X_eval))
    idx     = rng.choice(len(X_eval), size=n_eval, replace=False)
    phi     = np.zeros((n_eval, d))
    for si, x_ev in enumerate(X_eval[idx]):
        acc = np.zeros(d)
        for _ in range(args.M):
            perm   = _causal_perm(adj, rng)
            x_curr = X_train[rng.integers(n_tr)].copy()
            prev   = predict(x_curr.reshape(1,-1))[0]
            for f in perm:
                x_curr[f] = x_ev[f]
                nxt = predict(x_curr.reshape(1,-1))[0]
                acc[f] += gamma_est[f] * (nxt-prev)
                prev = nxt
        phi[si] = acc / args.M
    return phi, idx


#  MC SHAPLEY (reference for Wine/COMPAS)
#  Matches exactly the MCShapleyExplainer in the COMPAS notebook

def _build_classifier(X_train, y_train, task):
    """
    Returns (pred_proba, pred_cls_idx) where pred_cls_idx returns
    the 0-based COLUMN INDEX into pred_proba (not the raw class label).
    This handles wine classes {1,2,3} correctly.
    """
    if task == "binary" and HAS_XGB:
        from xgboost import XGBClassifier
        m = XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                          subsample=0.85, colsample_bytree=0.85,
                          random_state=args.seed, verbosity=0, n_jobs=4,
                          tree_method="hist", objective="binary:logistic")
        m.fit(X_train, y_train.astype(int))
        classes = m.classes_
        def pred_proba(Z): return m.predict_proba(Z)
        def pred_cls(Z):
            raw = m.predict(Z)
            return np.array([int(np.where(classes==r)[0][0]) for r in raw])
    else:
        sc = StandardScaler(); Xs = sc.fit_transform(X_train)
        m  = LogisticRegression(max_iter=1000,C=1.0).fit(Xs, y_train)
        classes = m.classes_
        def pred_proba(Z): return m.predict_proba(sc.transform(Z))
        def pred_cls(Z):
            raw = m.predict(sc.transform(Z))
            return np.array([int(np.where(classes==r)[0][0]) for r in raw])
    return pred_proba, pred_cls


def mc_shapley_global(X_train, y_train, X_test, task):
    """MC Shapley reference attribution (mean |phi| across K test instances)."""
    rng      = np.random.default_rng(args.seed)
    d        = X_train.shape[1]
    pred_proba, pred_cls = _build_classifier(X_train, y_train, task)
    K        = min(args.mc_k, len(X_test))
    selected = rng.choice(len(X_test), size=K, replace=False)
    local    = np.zeros((K, d))

    for cnt, i in enumerate(selected):
        if (cnt+1)%10==0: print(f"      MC Shapley {cnt+1}/{K} …", flush=True)
        x        = X_test[i].astype(float)
        pc       = int(pred_cls(x.reshape(1,-1))[0])
        bg_idx   = rng.choice(len(X_train),
                              size=min(args.mc_bg_size,len(X_train)), replace=False)
        bg       = X_train[bg_idx].copy()
        phi      = np.zeros(d)
        for _ in range(args.mc_iters):
            perm = rng.permutation(d); curr = []
            prev_v = float(np.mean(pred_proba(bg)[:,pc]))
            for p in perm:
                new_curr = curr + [int(p)]
                bg_mod   = bg.copy()
                for fi in new_curr: bg_mod[:,fi] = x[fi]
                new_v  = float(np.mean(pred_proba(bg_mod)[:,pc]))
                phi[p] += new_v - prev_v; curr = new_curr; prev_v = new_v
        local[cnt] = phi / args.mc_iters

    mean_abs = np.mean(np.abs(local), axis=0)
    return mean_abs / (mean_abs.sum()+1e-12), local, selected


#  METRICS (matching COMPAS notebook exactly)

def safe_spearman(a,b):
    a,b = np.asarray(a,float), np.asarray(b,float)
    if np.std(a)<1e-12 or np.std(b)<1e-12: return np.nan
    r,_ = spearmanr(a,b); return float(r)


def safe_pearson(a,b):
    a,b = np.asarray(a,float), np.asarray(b,float)
    if np.std(a)<1e-12 or np.std(b)<1e-12: return np.nan
    r,_ = pearsonr(a,b); return float(r)


def cosine_sim(a,b):
    a,b = np.asarray(a,float), np.asarray(b,float)
    denom = np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/denom) if denom>=1e-12 else np.nan


def normalize_abs(v):
    v = np.abs(np.asarray(v,float)); s = v.sum()
    return v/s if s>1e-12 else np.ones_like(v)/len(v)


def gt_metrics(mean_shap, gamma_gt):
    """Pearson/Spearman/Cosine/RMSE vs ground-truth gamma."""
    a = normalize_abs(mean_shap); g = normalize_abs(gamma_gt)
    return dict(
        pearson     = round(safe_pearson(a,g),4),
        spearman    = round(safe_spearman(a,g),4),
        cosine_sim  = round(cosine_sim(a,g),4),
        rmse        = round(float(np.sqrt(np.mean((a-g)**2))),6),
    )


def faithfulness_metrics(X_test, local_phi, eval_idx, pred_proba, pred_cls,
                         feat_means, d):
    """
    Faith_Spearman, Faith_Pearson, Faith_Cosine.
    local_phi[ri] is the causal SHAP attribution for X_test[eval_idx[ri]].
    Compare |local phi| vs |mean-sub drop| per instance, average across instances.
    Matches faithfulness_local_drop() in COMPAS notebook.
    """
    sp, pr, cs = [], [], []
    for ri in range(len(eval_idx)):
        i      = eval_idx[ri]
        x      = X_test[i].astype(float)
        pc_i   = int(pred_cls(x.reshape(1,-1))[0])
        base_p = float(pred_proba(x.reshape(1,-1))[0, pc_i])
        drops  = []
        for j in range(d):
            xm = x.copy(); xm[j] = feat_means[j]
            drops.append(base_p - float(pred_proba(xm.reshape(1,-1))[0, pc_i]))
        attr  = np.abs(local_phi[ri])
        drops = np.abs(np.array(drops, float))
        r = safe_spearman(attr, drops)
        if not np.isnan(r): sp.append(r)
        r = safe_pearson(attr, drops)
        if not np.isnan(r): pr.append(r)
        r = cosine_sim(attr, drops)
        if not np.isnan(r): cs.append(r)
    return (np.mean(sp) if sp else np.nan,
            np.mean(pr) if pr else np.nan,
            np.mean(cs) if cs else np.nan)


def robustness_metrics(X_train, X_test, y_train, adj, gamma_est, task,
                       feat_stds, pred_proba, pred_cls):
    """
    Robustness_Cosine, Rank_Robustness, L2:
    Matches robustness() in COMPAS notebook.
    """
    rng = np.random.default_rng(args.seed)
    K   = min(args.robust_k, len(X_test))
    sel = rng.choice(len(X_test), size=K, replace=False)
    cos_l, rank_l, l2_l = [],[],[]
    for k, i in enumerate(sel):
        x0      = X_test[i].astype(float)
        phi0,_  = _causal_shap_full(X_train, y_train,
                                    x0.reshape(1,-1), adj, gamma_est, task)
        phi0    = np.abs(phi0[0])
        for r in range(args.perturb_runs):
            noise = rng.normal(0, args.perturb_sigma, size=len(feat_stds)) * feat_stds
            xp    = (x0 + noise).reshape(1,-1)
            phip,_ = _causal_shap_full(X_train, y_train,
                                       xp, adj, gamma_est, task)
            phip  = np.abs(phip[0])
            cos_l.append(cosine_sim(phi0,phip))
            rk = safe_spearman(phi0,phip)
            if not np.isnan(rk): rank_l.append(rk)
            l2_l.append(float(np.linalg.norm(phi0-phip)))
    return (np.mean(cos_l)  if cos_l  else np.nan,
            np.mean(rank_l) if rank_l else np.nan,
            np.mean(l2_l)   if l2_l   else np.nan)


#  GRAPH  (same aesthetic as before)

POS_COL=  '#1B8C7A'; NEG_COL='#C0392B'; EST_COL='#8E24AA'
TARGET_COL='#6A1B9A'; BLUES=['#AED6F1','#5DADE2','#2E86C1','#1B5E87','#0D3349']

def _wrap(name, maxlen=12):
    parts=name.split()
    if len(name)<=maxlen or len(parts)==1: return name
    mid=len(parts)//2
    return ' '.join(parts[:mid])+'\n'+' '.join(parts[mid:])


def draw_causal_graph(adj_feat, feat_names, gamma_est, target_name, title, out_path):
    d=len(feat_names)
    G=nx.DiGraph(); G.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if abs(adj_feat[i,j])>1e-6: G.add_edge(i,j,weight=float(adj_feat[i,j]))
    try: layers=list(nx.topological_generations(G))
    except: layers=[list(range(d))]
    pos={}; nl=len(layers)
    for li,layer in enumerate(layers):
        layer=sorted(layer); n=len(layer)
        y=1.0-li*(0.80/max(nl-1,1))
        for ki,node in enumerate(layer): pos[node]=((ki+1)/(n+1),y)
    TARGET=d; pos[TARGET]=(0.5,-0.08)
    figw=max(12,d*1.3); figh=max(10,(nl+2)*1.8)
    fig,ax=plt.subplots(figsize=(figw,figh))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')
    feat_labels=[]
    for src,dst,data in G.edges(data=True):
        w=data['weight']; col=POS_COL if w>0 else NEG_COL
        lw=0.9+min(abs(w)*2.5,4.0)
        xi,yi=pos[src]; xj,yj=pos[dst]
        dy_abs=abs(yi-yj)
        rad=0.0 if dy_abs>0.35 else (0.25 if dy_abs<0.05 else 0.12)
        ax.annotate("",xy=(xj,yj),xytext=(xi,yi),
            arrowprops=dict(arrowstyle="-|>",color=col,lw=lw,alpha=0.72,
                            connectionstyle=f"arc3,rad={rad}",shrinkA=23,shrinkB=23))
        t=0.42; mx=xi+t*(xj-xi); my=yi+t*(yj-yi)
        dx,dy=xj-xi,yj-yi; norm=max(np.hypot(dx,dy),1e-6)
        side=1 if (src+dst)%2==0 else -1
        ox=side*(-dy/norm)*0.04; oy=side*(dx/norm)*0.04
        feat_labels.append((mx+ox,my+oy,f"{w:+.2f}",'white','#cccccc'))
    feat_sorted=sorted(range(d),key=lambda i:pos[i][0])
    tx,ty=pos[TARGET]; tgt_labels=[]
    for rank,fi in enumerate(feat_sorted):
        w=float(gamma_est[fi])
        if w<1e-6: continue
        xi,yi=pos[fi]; lw=0.8+min(w*30,3.5)
        spread=min(0.28,d*0.04); arr_x=tx+spread*(rank/(max(d-1,1))-0.5)
        ax.annotate("",xy=(arr_x,ty+0.065),xytext=(xi,yi),
            arrowprops=dict(arrowstyle="-|>",color=EST_COL,lw=lw,alpha=0.60,
                            linestyle='dashed',connectionstyle="arc3,rad=0.0",
                            shrinkA=23,shrinkB=26))
        t=0.55; mx=xi+t*(arr_x-xi); my=yi+t*(ty+0.065-yi)
        dx,dy=arr_x-xi,ty+0.065-yi; norm=max(np.hypot(dx,dy),1e-6)
        side=1 if fi%2==0 else -1
        ox=side*(-dy/norm)*0.035; oy=side*(dx/norm)*0.035
        tgt_labels.append((mx+ox,my+oy,f"{w:.3f}"))
    for lx,ly,lab,fc,ec in feat_labels:
        ax.text(lx,ly,lab,ha='center',va='center',fontsize=7.5,color='#111111',
                fontweight='medium',zorder=8,
                bbox=dict(boxstyle='round,pad=0.18',fc=fc,ec=ec,alpha=0.90,lw=0.5))
    for lx,ly,lab in tgt_labels:
        ax.text(lx,ly,lab,ha='center',va='center',fontsize=7.5,color='#4A148C',
                fontweight='semibold',zorder=10,
                bbox=dict(boxstyle='round,pad=0.15',fc='#F3E5F5',ec='#CE93D8',alpha=0.93,lw=0.5))
    in_deg=dict(G.in_degree())
    for i in range(d):
        x,y=pos[i]; col=BLUES[min(in_deg.get(i,0),len(BLUES)-1)]
        label=_wrap(feat_names[i]); lines=label.count('\n')+1
        bw=max(0.10,len(max(label.split('\n'),key=len))*0.013)
        bh=0.055+(lines-1)*0.033
        ax.add_patch(mpatches.FancyBboxPatch((x-bw,y-bh/2),bw*2,bh,
            boxstyle="round,pad=0.015",facecolor=col,edgecolor='white',linewidth=2,zorder=5))
        ax.text(x,y,label,ha='center',va='center',fontsize=8.5,color='white',
                fontweight='bold',zorder=6,multialignment='center',linespacing=1.25)
    tbw=max(0.14,len(target_name)*0.013)
    ax.add_patch(mpatches.FancyBboxPatch((tx-tbw,ty-0.065),tbw*2,0.13,
        boxstyle="round,pad=0.018",facecolor=TARGET_COL,edgecolor='white',linewidth=2.5,zorder=5))
    ax.text(tx,ty,target_name,ha='center',va='center',fontsize=10,color='white',
            fontweight='bold',zorder=6)
    handles=[
        plt.Line2D([0],[0],color=POS_COL,lw=2,label='Positive inter-feature effect (DrBO)'),
        plt.Line2D([0],[0],color=NEG_COL,lw=2,label='Negative inter-feature effect (DrBO)'),
        plt.Line2D([0],[0],color=EST_COL,lw=2,ls='--',
                   label='Feature → target  (γ_est, OLS causal effect)'),
        mpatches.Patch(facecolor=BLUES[3],label='High in-degree feature'),
        mpatches.Patch(facecolor=BLUES[0],label='Low in-degree feature'),
        mpatches.Patch(facecolor=TARGET_COL,label='Target variable'),
    ]
    ax.legend(handles=handles,loc='lower left',fontsize=8.5,
              framealpha=0.95,edgecolor='#cccccc',fancybox=True,ncol=2)
    ax.set_title(title,fontsize=14,fontweight='bold',pad=16,color='#111111')
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.28,1.25); ax.axis('off')
    fig.tight_layout(pad=2.2)
    fig.savefig(out_path,dpi=300,bbox_inches='tight',facecolor='white')
    plt.close(); print(f"    graph img  → {out_path}")


def save_graph_files(adj, feat_names, gamma_est, target_name, dname):
    tag=f"{dname}_drbo"; d=len(feat_names)
    pd.DataFrame(adj,index=feat_names,columns=feat_names).to_csv(
        os.path.join(OUT_DIR,f"{tag}_adj_matrix.csv"))
    rows=[]
    for i in range(d):
        for j in range(d):
            if abs(adj[i,j])>1e-6:
                rows.append({"source":feat_names[i],"target_node":feat_names[j],
                             "weight":round(adj[i,j],6),"edge_type":"inter_feature"})
    for i,f in enumerate(feat_names):
        rows.append({"source":f,"target_node":target_name,
                     "weight":round(float(gamma_est[i]),6),
                     "edge_type":"feature_to_target_gamma_est"})
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,f"{tag}_graph_edges.csv"),index=False)
    draw_causal_graph(adj,feat_names,gamma_est,target_name,
                      f"DrBO Causal Graph — {target_name}",
                      os.path.join(GRAPH_DIR,f"{tag}_graph.png"))


# Bar chart

def plot_bar(mean_shap, feat_names, dname, gamma_gt=None, mc_ref=None):
    cmap=plt.get_cmap("tab10"); colors=[cmap(i%10) for i in range(len(feat_names))]
    fig,ax=plt.subplots(figsize=(max(10,len(feat_names)*0.8),5))
    ax.bar(feat_names,mean_shap,color=colors,edgecolor='none')
    if gamma_gt is not None and len(gamma_gt)==len(feat_names):
        gn=gamma_gt/(gamma_gt.sum()+1e-12)*mean_shap.sum()
        ax.plot(feat_names,gn,color='black',lw=1.8,ls='--',
                marker='o',ms=5,label='Ground Truth γ (SEM)',zorder=5)
    if mc_ref is not None:
        mc_n=normalize_abs(mc_ref)*mean_shap.sum()
        ax.plot(feat_names,mc_n,color='crimson',lw=1.8,ls=':',
                marker='s',ms=5,label='MC Shapley reference',zorder=5)
    if gamma_gt is not None or mc_ref is not None: ax.legend(fontsize=10)
    ax.set_xlabel("Feature",fontsize=11); ax.set_ylabel("Mean |Causal SHAP|",fontsize=11)
    ax.set_title(f"Causal SHAP (DrBO) — {dname}",fontsize=12,fontweight='bold')
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names,rotation=90,ha='center',fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    fig.tight_layout()
    out=os.path.join(BARCHART_DIR,f"{dname}_drbo_causal_shap.png")
    fig.savefig(out,dpi=600,bbox_inches='tight'); plt.close()
    print(f"    bar chart  → {out}")


# Per-dataset pipeline

def run_dataset(loader):
    try:
        out = loader()
    except FileNotFoundError as e:
        print(f"  [SKIP] {e}"); return None
    X_train,X_test,y_train,y_test,gamma_gt,feat_names,task,dname,target_name = out

    n,d=X_train.shape
    has_gt = gamma_gt is not None and len(gamma_gt)==d
    print(f"\n{'='*60}")
    print(f"  {dname.upper()}  n_train={n}  d={d}  task={task}")
    if has_gt:
        print(f"  γ_gt: { {feat_names[i]:round(float(gamma_gt[i]),4) for i in range(d)} }")
    else:
        print(f"  γ_gt: None → metrics vs MC Shapley + faithfulness + robustness")
    print(f"{'='*60}")

    X_sc_train = StandardScaler().fit_transform(X_train)

    # 1. Causal discovery
    print("  [1/5] Causal discovery (DrBO, features only) …")
    adj, t_disc = discover(X_sc_train, dname)

    # 2. γ_est
    print("  [2/5] Estimating γ_est via OLS …")
    gamma_est = estimate_gamma(X_train, y_train, task)
    print(f"    γ_est: { {feat_names[i]:round(float(gamma_est[i]),4) for i in range(d)} }")

    # 3. Causal SHAP
    print("  [3/5] Causal SHAP (γ_est-weighted) …")
    t1=time.time()
    phi,eval_idx = _causal_shap_full(X_train,y_train,X_test,adj,gamma_est,task)
    t_shap=time.time()-t1; mean_shap=np.mean(np.abs(phi),axis=0)
    shap_df=pd.DataFrame(phi,columns=feat_names)
    shap_df.insert(0,"sample_idx",eval_idx)
    shap_df.to_csv(os.path.join(OUT_DIR,f"{dname}_drbo_causal_shap_values.csv"),index=False)
    print(f"    done {t_shap:.1f}s")

    # 4. Graph
    print("  [4/5] Saving graph …")
    save_graph_files(adj,feat_names,gamma_est,target_name,dname)

    # 5. Metrics
    print("  [5/5] Metrics & charts …")
    row=dict(dataset=dname,n_train=n,n_features=d,task=task,target=target_name,
             n_inter_edges=int((np.abs(adj)>1e-6).sum()),
             time_disc_s=round(t_disc,2),time_shap_s=round(t_shap,2))

    mc_ref_attr = None

    if has_gt:
        # ── Datasets WITH ground truth: Pearson/Spearman/Cosine/RMSE vs γ_gt ──
        m=gt_metrics(mean_shap,gamma_gt); row.update(m)
        row.update(faith_spearman=np.nan,faith_pearson=np.nan,faith_cosine=np.nan,
                   pred_align_mc=np.nan,attr_cosine_mc=np.nan,
                   robustness_cosine=np.nan,rank_robustness=np.nan,l2=np.nan)
        print(f"    SHAP vs γ_gt: {m}")
        plot_bar(mean_shap,feat_names,dname,gamma_gt=gamma_gt)

    else:
        # ── Wine / COMPAS: full metric suite from COMPAS notebook ─────────────
        row.update(pearson=np.nan,spearman=np.nan,cosine_sim=np.nan,rmse=np.nan)

        # MC Shapley reference
        print("    Computing MC Shapley reference …")
        mc_ref_attr,mc_local,mc_sel = mc_shapley_global(X_train,y_train,X_test,task)

        # Faithfulness (against the Causal SHAP local explanations)
        pred_proba,pred_cls = _build_classifier(X_train,y_train,task)
        feat_means = X_train.mean(axis=0)
        faith_sp,faith_pr,faith_cs = faithfulness_metrics(
            X_test,phi,eval_idx,pred_proba,pred_cls,feat_means,d)

        # Alignment vs MC Shapley
        an = normalize_abs(mean_shap); ref = normalize_abs(mc_ref_attr)
        pred_align = safe_pearson(an,ref)
        attr_cos   = cosine_sim(an,ref)

        # Robustness
        feat_stds = X_train.std(axis=0)+1e-8
        rob_cos,rank_rob,l2 = robustness_metrics(
            X_train,X_test,y_train,adj,gamma_est,task,feat_stds,pred_proba,pred_cls)

        row.update(faith_spearman=round(faith_sp,4) if not np.isnan(faith_sp) else np.nan,
                   faith_pearson =round(faith_pr,4) if not np.isnan(faith_pr) else np.nan,
                   faith_cosine  =round(faith_cs,4) if not np.isnan(faith_cs) else np.nan,
                   pred_align_mc =round(pred_align,4) if not np.isnan(pred_align) else np.nan,
                   attr_cosine_mc=round(attr_cos,4),
                   robustness_cosine=round(rob_cos,4) if not np.isnan(rob_cos) else np.nan,
                   rank_robustness  =round(rank_rob,4) if not np.isnan(rank_rob) else np.nan,
                   l2               =round(l2,4) if not np.isnan(l2) else np.nan)
        print(f"    Faith: Sp={faith_sp:.4f}  Pr={faith_pr:.4f}  Cos={faith_cs:.4f}")
        print(f"    vs MC: align={pred_align:.4f}  cosine={attr_cos:.4f}")
        print(f"    Robustness: cos={rob_cos:.4f}  rank={rank_rob:.4f}  L2={l2:.4f}")
        plot_bar(mean_shap,feat_names,dname,mc_ref=mc_ref_attr)

    pd.DataFrame([row]).to_csv(
        os.path.join(OUT_DIR,f"{dname}_drbo_metrics.csv"),index=False)

    # Feature importance table
    imp=pd.DataFrame({"feature":feat_names,"mean_abs_shap":mean_shap,
                      "gamma_est":gamma_est,
                      "gamma_gt":gamma_gt if has_gt else [np.nan]*d})
    imp=imp.sort_values("mean_abs_shap",ascending=False).reset_index(drop=True)
    imp.to_csv(os.path.join(OUT_DIR,f"{dname}_drbo_feature_importance.csv"),index=False)
    print(f"\n  Feature importance ({dname}):\n{imp.to_string(index=False)}")
    return row


# Summary

def plot_summary_gt(df):
    ms=["pearson","spearman","cosine_sim","rmse"]
    dfp=df.dropna(subset=ms)
    if dfp.empty: return
    cmap=plt.get_cmap("tab10")
    fig,axes=plt.subplots(1,4,figsize=(22,5))
    for ax,m in zip(axes,ms):
        v=dfp[m].values; labels=dfp["dataset"].values
        ax.bar(labels,v,color=[cmap(i%10) for i in range(len(labels))],edgecolor='none')
        ax.set_title(m,fontsize=12,fontweight='bold')
        ax.set_xticklabels(labels,rotation=90,fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        if m!="rmse": ax.set_ylim(-1,1)
    fig.suptitle("Causal SHAP (DrBO) vs Ground Truth γ_gt",fontsize=13,fontweight='bold')
    fig.tight_layout()
    out=os.path.join(BARCHART_DIR,"summary_gt_metrics.png")
    fig.savefig(out,dpi=600,bbox_inches='tight'); plt.close()
    print(f"  GT summary → {out}")


def plot_summary_mc(df):
    ms=["faith_spearman","faith_pearson","faith_cosine",
        "pred_align_mc","attr_cosine_mc","robustness_cosine","rank_robustness","l2"]
    dfp=df.dropna(subset=["faith_spearman"])
    if dfp.empty: return
    cmap=plt.get_cmap("tab10")
    fig,axes=plt.subplots(2,4,figsize=(22,10))
    for ax,m in zip(axes.flat,ms):
        v=dfp[m].values if m in dfp.columns else []
        labels=dfp["dataset"].values
        if len(v):
            ax.bar(labels,v,color=[cmap(i%10) for i in range(len(labels))],edgecolor='none')
        ax.set_title(m,fontsize=11,fontweight='bold')
        ax.set_xticklabels(labels,rotation=90,fontsize=9)
        ax.spines[['top','right']].set_visible(False)
    fig.suptitle("Causal SHAP (DrBO) — Wine/COMPAS metrics vs MC Shapley",
                 fontsize=13,fontweight='bold')
    fig.tight_layout()
    out=os.path.join(BARCHART_DIR,"summary_mc_metrics.png")
    fig.savefig(out,dpi=600,bbox_inches='tight'); plt.close()
    print(f"  MC summary → {out}")


def main():
    print("\n"+"="*65)
    print("  Causal SHAP + DrBO  |  6 Datasets")
    print(f"  data_dir={DATA_DIR}  n_evals={args.n_evals}")
    print(f"  n_shap_samples={args.n_shap_samples}  M={args.M}")
    print("="*65)
    rows=[r for loader in LOADERS if (r:=run_dataset(loader)) is not None]
    if rows:
        summary=pd.DataFrame(rows)
        summary.to_csv(os.path.join(OUT_DIR,"all_datasets_summary.csv"),index=False)
        print(f"\n{'='*65}\nSUMMARY\n{'='*65}")
        print(summary.to_string(index=False))
        plot_summary_gt(summary)
        plot_summary_mc(summary)
    print("\nAll done.")

if __name__=="__main__":
    main()

[OK] DrBO loaded.

  Causal SHAP + DrBO  |  6 Datasets
  data_dir=../datasets  n_evals=20000
  n_shap_samples=200  M=512

  WINE  n_train=141  d=13  task=multiclass
  γ_gt: None → metrics vs MC Shapley + faithfulness + robustness
  [1/5] Causal discovery (DrBO, features only) …
    Running DrBO (20000 evals) — this may take several minutes …
  [DrBO] wine: 29 edges, 1330.3s
  [2/5] Estimating γ_est via OLS …
    γ_est: {'Alcohol': 0.1027, 'Malic acid': 0.0548, 'Ash': 0.0907, 'Alcalinity of ash': 0.0993, 'Magnesium': 0.0167, 'Total phenols': 0.0344, 'Flavanoids': 0.109, 'Nonflavanoid phenols': 0.0133, 'Proanthocyanins': 0.0601, 'Color intensity': 0.1287, 'Hue': 0.0866, 'OD280/OD315 of diluted wines': 0.0751, 'Proline': 0.1285}
  [3/5] Causal SHAP (γ_est-weighted) …
    done 38.7s
  [4/5] Saving graph …
    graph img  → results_drbo/graphs/wine_drbo_graph.png
  [5/5] Metrics & charts …
    Computing MC Shapley reference …
      MC Shapley 10/36 …
      MC Shapley 20/36 …
      MC Shapley